In [ ]:

# Email Phishing Detection - Feature Extraction

## Objective
To convert cleaned email text into numerical features that a machine learning model can understand.

## Features Used
1. TF-IDF text features
2. Text length
3. Word count
4. URL count
5. Urgency word count
6. Threat word count
7. Financial word count

## Input
Cleaned email dataset generated by the data cleaning notebook.

## Output
Numerical feature matrices for model training and testing.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Project folder
PROJECT_ROOT = Path(r"C:\Users\nijag\Desktop\Anjaliiiii")

# Processed email data folder
PROCESSED_EMAIL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "email"
)

# Cleaned dataset saved by the cleaning notebook
CLEANED_DATA_PATH = (
    PROCESSED_EMAIL_DIR / "email_cleaned.csv"
)

print("Project setup complete.")
print("Cleaned dataset exists:", CLEANED_DATA_PATH.exists())
print("Cleaned dataset path:", CLEANED_DATA_PATH)

Project setup complete.
Cleaned dataset exists: True
Cleaned dataset path: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\email_cleaned.csv


In [2]:

# Load the cleaned email dataset
combined_email = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned email dataset loaded successfully.")
print("Dataset shape:", combined_email.shape)
print("\nAvailable columns:")
print(combined_email.columns.tolist())

# Preview the first five rows
combined_email.head()

Cleaned email dataset loaded successfully.
Dataset shape: (164564, 9)

Available columns:
['subject', 'body', 'sender', 'receiver', 'date', 'urls', 'label', 'source', 'extracted_url_count']


,subject,body,sender,receiver,date,urls,label,source,extracted_url_count
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",1.0,1,CEAS_08,1
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",1.0,1,CEAS_08,1
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",1.0,1,CEAS_08,24
3,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",1.0,0,CEAS_08,652
4,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",1.0,1,CEAS_08,1


In [12]:

# Verify the cleaned dataset before feature extraction

print("Dataset shape:", combined_email.shape)
print("Email text column exists:", "email_text" in combined_email.columns)

print(
    "Empty-text emails:",
    (combined_email["email_text"].fillna("").str.strip() == "").sum()
)

print("\nDataset preview:")
combined_email[["subject", "body", "email_text", "label"]].head()

Dataset shape: (164563, 10)
Email text column exists: True
Empty-text emails: 0

Dataset preview:


,subject,body,email_text,label
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...","Never agree to be a loser Buck up, your troubl...",1
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,Befriend Jenna Jameson \nUpgrade your sex and ...,1
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,1
3,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,Re: svn commit: r619753 - in /spamassassin/tru...,0
4,SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,SpecialPricesPharmMoreinfo \nWelcomeFastShippi...,1


In [4]:

# Separate email text (input) and labels (target)
X = combined_email["email_text"]
y = combined_email["label"].astype(int)

print("Input text (X) shape:", X.shape)
print("Target labels (y) shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts().sort_index())

print("\nMissing input values:", X.isna().sum())
print("Missing target values:", y.isna().sum())

Input text (X) shape: (164563,)
Target labels (y) shape: (164563,)

Label distribution:
label
0    78828
1    85735
Name: count, dtype: int64

Missing input values: 0
Missing target values: 0


In [5]:

from sklearn.model_selection import train_test_split

# Split the dataset: 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

Training samples: 131650
Testing samples: 32913

Training label distribution:
label
0    63062
1    68588
Name: count, dtype: int64

Testing label distribution:
label
0    15766
1    17147
Name: count, dtype: int64


In [6]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

# Learn vocabulary and extract features from training emails
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform testing emails using the same learned vocabulary
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF feature extraction completed!")
print("Training TF-IDF matrix:", X_train_tfidf.shape)
print("Testing TF-IDF matrix:", X_test_tfidf.shape)
print("Number of learned features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF feature extraction completed!
Training TF-IDF matrix: (131650, 20000)
Testing TF-IDF matrix: (32913, 20000)
Number of learned features: 20000


In [7]:

# Create numerical feature tables for training and testing
train_features = pd.DataFrame(index=X_train.index)
test_features = pd.DataFrame(index=X_test.index)

# 1. Total characters in each email
train_features["text_length"] = X_train.str.len()
test_features["text_length"] = X_test.str.len()

# 2. Number of words in each email
train_features["word_count"] = X_train.str.split().str.len()
test_features["word_count"] = X_test.str.split().str.len()

# 3. Number of URLs in each email
url_pattern = r'https?://\S+|www\.\S+'

train_features["url_count"] = X_train.str.count(url_pattern)
test_features["url_count"] = X_test.str.count(url_pattern)

print("Basic numerical features extracted!")
print("Training numerical features:", train_features.shape)
print("Testing numerical features:", test_features.shape)

print("\nSample training features:")
print(train_features.head())

Basic numerical features extracted!
Training numerical features: (131650, 3)
Testing numerical features: (32913, 3)

Sample training features:
        text_length  word_count  url_count
156678         1579         233          0
3406            378          68          0
38460           990         146          1
77834            92          18          0
30406          3899         310         24


In [8]:

import re

# Keyword groups
urgency_words = [
    "urgent", "immediately", "now", "hurry",
    "expires", "limited time", "act now"
]

threat_words = [
    "suspended", "blocked", "terminated",
    "locked", "deactivated", "unauthorized"
]

financial_words = [
    "bank", "payment", "account", "invoice",
    "refund", "credit card", "transaction", "money"
]

# Count occurrences of words and phrases
def count_word_group(text, word_list):
    text = str(text).lower()

    return sum(
        len(re.findall(r"\b" + re.escape(word) + r"\b", text))
        for word in word_list
    )

# Add keyword-count features to the training data
train_features["urgency_count"] = X_train.apply(
    lambda text: count_word_group(text, urgency_words)
)
train_features["threat_count"] = X_train.apply(
    lambda text: count_word_group(text, threat_words)
)
train_features["financial_count"] = X_train.apply(
    lambda text: count_word_group(text, financial_words)
)

# Add the same features to the testing data
test_features["urgency_count"] = X_test.apply(
    lambda text: count_word_group(text, urgency_words)
)
test_features["threat_count"] = X_test.apply(
    lambda text: count_word_group(text, threat_words)
)
test_features["financial_count"] = X_test.apply(
    lambda text: count_word_group(text, financial_words)
)

print("Keyword-count features extracted!")
print("Training numerical features:", train_features.shape)
print("Testing numerical features:", test_features.shape)

print("\nSample training features:")
print(train_features.head())

Keyword-count features extracted!
Training numerical features: (131650, 6)
Testing numerical features: (32913, 6)

Sample training features:
        text_length  word_count  url_count  urgency_count  threat_count  \
156678         1579         233          0              0             0   
3406            378          68          0              0             0   
38460           990         146          1              1             0   
77834            92          18          0              0             0   
30406          3899         310         24              0             0   

        financial_count  
156678                8  
3406                  0  
38460                 0  
77834                 0  
30406                 0  


In [9]:

from scipy.sparse import hstack, csr_matrix

# Convert numerical feature tables to sparse matrices
train_numeric_sparse = csr_matrix(train_features.values)
test_numeric_sparse = csr_matrix(test_features.values)

# Combine TF-IDF and numerical features
X_train_combined = hstack(
    [X_train_tfidf, train_numeric_sparse],
    format="csr"
)

X_test_combined = hstack(
    [X_test_tfidf, test_numeric_sparse],
    format="csr"
)

print("Feature matrices combined successfully!")
print("Training feature matrix:", X_train_combined.shape)
print("Testing feature matrix:", X_test_combined.shape)

print("\nTotal TF-IDF features:", X_train_tfidf.shape[1])
print("Total numerical features:", train_features.shape[1])
print("Total features per email:", X_train_combined.shape[1])

Feature matrices combined successfully!
Training feature matrix: (131650, 20006)
Testing feature matrix: (32913, 20006)

Total TF-IDF features: 20000
Total numerical features: 6
Total features per email: 20006


In [10]:

# Final validation before saving the feature-extraction results

print("TRAINING DATA")
print("Feature matrix shape:", X_train_combined.shape)
print("Labels shape:", y_train.shape)
print("Missing labels:", y_train.isna().sum())

print("\nTESTING DATA")
print("Feature matrix shape:", X_test_combined.shape)
print("Labels shape:", y_test.shape)
print("Missing labels:", y_test.isna().sum())

print("\nNUMERICAL FEATURES")
print("Training features missing values:",
      train_features.isna().sum().sum())
print("Testing features missing values:",
      test_features.isna().sum().sum())

# Check whether dimensions match
assert X_train_combined.shape[0] == len(y_train)
assert X_test_combined.shape[0] == len(y_test)
assert X_train_combined.shape[1] == X_test_combined.shape[1]
assert train_features.isna().sum().sum() == 0
assert test_features.isna().sum().sum() == 0

print("\nAll validation checks passed!")

TRAINING DATA
Feature matrix shape: (131650, 20006)
Labels shape: (131650,)
Missing labels: 0

TESTING DATA
Feature matrix shape: (32913, 20006)
Labels shape: (32913,)
Missing labels: 0

NUMERICAL FEATURES
Training features missing values: 0
Testing features missing values: 0

All validation checks passed!


In [11]:

from scipy.sparse import save_npz
import joblib

# Create a folder to store extracted features
FEATURES_DIR = PROCESSED_EMAIL_DIR / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Save training and testing feature matrices
save_npz(
    FEATURES_DIR / "X_train_features.npz",
    X_train_combined
)
save_npz(
    FEATURES_DIR / "X_test_features.npz",
    X_test_combined
)

# Save the corresponding labels
y_train.to_csv(FEATURES_DIR / "y_train.csv", index=False)
y_test.to_csv(FEATURES_DIR / "y_test.csv", index=False)

# Save the fitted TF-IDF vectorizer
joblib.dump(
    tfidf_vectorizer,
    FEATURES_DIR / "tfidf_vectorizer.joblib"
)

# Save the numerical feature names and their order
joblib.dump(
    train_features.columns.tolist(),
    FEATURES_DIR / "numerical_feature_names.joblib"
)

print("Feature-extraction outputs saved successfully!")
print("Saved folder:", FEATURES_DIR)

print("\nFiles saved:")
for file_path in sorted(FEATURES_DIR.iterdir()):
    print("-", file_path.name)

Feature-extraction outputs saved successfully!
Saved folder: C:\Users\nijag\Desktop\Anjaliiiii\data\processed\email\features

Files saved:
- numerical_feature_names.joblib
- tfidf_vectorizer.joblib
- X_test_features.npz
- X_train_features.npz
- y_test.csv
- y_train.csv


In [13]:

# STEP 7: Separate email text and labels

# X contains the email text that the model will learn from
X = combined_email["email_text"]

# y contains the correct answer for each email
y = combined_email["label"].astype(int)

print("Email text (X) shape:", X.shape)
print("Labels (y) shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts().sort_index())

print("\nLabel meanings:")
print("0 = Legitimate email")
print("1 = Phishing email")

Email text (X) shape: (164563,)
Labels (y) shape: (164563,)

Label distribution:
label
0    78828
1    85735
Name: count, dtype: int64

Label meanings:
0 = Legitimate email
1 = Phishing email


In [14]:

# STEP 8: Split data into training and testing sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data splitting completed successfully!")

print("\nTraining data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

Data splitting completed successfully!

Training data:
X_train: (131650,)
y_train: (131650,)

Testing data:
X_test: (32913,)
y_test: (32913,)

Training label distribution:
label
0    63062
1    68588
Name: count, dtype: int64

Testing label distribution:
label
0    15766
1    17147
Name: count, dtype: int64


In [15]:

# STEP 9: Convert email text into TF-IDF features

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

# Learn vocabulary and transform training emails
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform testing emails using the same learned vocabulary
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF conversion completed successfully!")

print("\nTraining TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

print("\nNumber of words and phrases learned:",
      len(tfidf_vectorizer.vocabulary_))

print("\nExample learned terms:")
print(list(tfidf_vectorizer.vocabulary_.keys())[:20])

TF-IDF conversion completed successfully!

Training TF-IDF shape: (131650, 20000)
Testing TF-IDF shape: (32913, 20000)

Number of words and phrases learned: 20000

Example learned terms:
['hello', 'letter', 'may', 'come', 'surprise', 'due', 'fact', 'yet', 'met', 'say', 'intentions', 'causing', 'decided', 'contact', 'medium', 'read', 'dont', 'want', 'feel', 'sorry']


In [16]:

# STEP 10: Extract numerical email features

import re
import pandas as pd

# Words associated with urgency, threats, and financial activity
urgency_words = [
    "urgent", "immediately", "now", "hurry",
    "expires", "limited time", "act now"
]

threat_words = [
    "suspended", "blocked", "terminated",
    "locked", "deactivated", "unauthorized"
]

financial_words = [
    "bank", "payment", "account", "invoice",
    "refund", "credit card", "transaction", "money"
]

# Count occurrences of these words or phrases
def count_keywords(text, keywords):
    text = str(text).lower()
    count = 0

    for word in keywords:
        pattern = r"\b" + re.escape(word) + r"\b"
        count += len(re.findall(pattern, text))

    return count

# Extract six numerical features from each email
def extract_email_features(text):
    text = str(text)

    return {
        "text_length": len(text),
        "word_count": len(text.split()),
        "url_count": len(
            re.findall(r"https?://\S+|www\.\S+", text)
        ),
        "urgency_count": count_keywords(text, urgency_words),
        "threat_count": count_keywords(text, threat_words),
        "financial_count": count_keywords(text, financial_words)
    }

# Apply feature extraction separately to training and testing emails
train_features = pd.DataFrame(
    X_train.apply(extract_email_features).tolist(),
    index=X_train.index
)

test_features = pd.DataFrame(
    X_test.apply(extract_email_features).tolist(),
    index=X_test.index
)

print("Numerical feature extraction completed!")

print("\nTraining numerical features:", train_features.shape)
print("Testing numerical features:", test_features.shape)

print("\nFeature names:")
print(train_features.columns.tolist())

print("\nSample training features:")
display(train_features.head())

Numerical feature extraction completed!

Training numerical features: (131650, 6)
Testing numerical features: (32913, 6)

Feature names:
['text_length', 'word_count', 'url_count', 'urgency_count', 'threat_count', 'financial_count']

Sample training features:


,text_length,word_count,url_count,urgency_count,threat_count,financial_count
156678,1579,233,0,0,0,8
3406,378,68,0,0,0,0
38460,990,146,1,1,0,0
77834,92,18,0,0,0,0
30406,3899,310,24,0,0,0


In [18]:
# Check whether TF-IDF variables already exist

tfidf_variables = [
    "tfidf_vectorizer",
    "X_tfidf_train",
    "X_tfidf_test",
    "X_train_tfidf",
    "X_test_tfidf"
]

for variable in tfidf_variables:
    print(f"{variable}: {variable in globals()}")

tfidf_vectorizer: True
X_tfidf_train: False
X_tfidf_test: False
X_train_tfidf: True
X_test_tfidf: True


In [19]:

# Verify the existing TF-IDF features

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

print("\nTraining matrix type:", type(X_train_tfidf))
print("Testing matrix type:", type(X_test_tfidf))

print("\nVocabulary size:", len(tfidf_vectorizer.vocabulary_))

# Confirm both matrices use the same feature columns
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]

print("\nSUCCESS: TF-IDF training and testing matrices are compatible.")

TF-IDF training shape: (131650, 20000)
TF-IDF testing shape: (32913, 20000)

Training matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Testing matrix type: <class 'scipy.sparse._csr.csr_matrix'>

Vocabulary size: 20000

SUCCESS: TF-IDF training and testing matrices are compatible.


In [20]:

# Check existing numerical feature and label variables

variables_to_check = [
    "X_num_train",
    "X_num_test",
    "X_train_num",
    "X_test_num",
    "numerical_features",
    "y_train",
    "y_test",
    "X_train_tfidf",
    "X_test_tfidf"
]

print("Checking existing variables:\n")

for name in variables_to_check:
    if name in globals():
        value = globals()[name]
        print(f"{name}: exists | shape = {getattr(value, 'shape', 'N/A')}")
    else:
        print(f"{name}: not found")

Checking existing variables:

X_num_train: not found
X_num_test: not found
X_train_num: not found
X_test_num: not found
numerical_features: not found
y_train: exists | shape = (131650,)
y_test: exists | shape = (32913,)
X_train_tfidf: exists | shape = (131650, 20000)
X_test_tfidf: exists | shape = (32913, 20000)


In [22]:

# Safely search for existing numerical feature variables

for name, value in list(globals().items()):
    if name.startswith("_"):
        continue

    shape = getattr(value, "shape", None)

    if isinstance(shape, tuple) and len(shape) == 2:
        if shape[0] in (131650, 32913):
            print(f"{name}: shape = {shape}")

X_train_tfidf: shape = (131650, 20000)
X_test_tfidf: shape = (32913, 20000)
train_features: shape = (131650, 6)
test_features: shape = (32913, 6)
train_numeric_sparse: shape = (131650, 6)
test_numeric_sparse: shape = (32913, 6)
X_train_combined: shape = (131650, 20006)
X_test_combined: shape = (32913, 20006)


In [23]:

# Final feature-combination verification

print("Training features:", X_train_combined.shape)
print("Testing features:", X_test_combined.shape)

print("\nTraining labels:", y_train.shape)
print("Testing labels:", y_test.shape)

assert X_train_combined.shape[0] == len(y_train)
assert X_test_combined.shape[0] == len(y_test)
assert X_train_combined.shape[1] == X_test_combined.shape[1]

print("\nSUCCESS: Combined features and labels have matching dimensions.")

Training features: (131650, 20006)
Testing features: (32913, 20006)

Training labels: (131650,)
Testing labels: (32913,)

SUCCESS: Combined features and labels have matching dimensions.


In [24]:

# Check the distribution of labels in training and testing data

import pandas as pd

print("Training label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

print("\nTraining label percentages:")
print((pd.Series(y_train).value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTesting label percentages:")
print((pd.Series(y_test).value_counts(normalize=True).sort_index() * 100).round(2))

Training label distribution:
label
0    63062
1    68588
Name: count, dtype: int64

Testing label distribution:
label
0    15766
1    17147
Name: count, dtype: int64

Training label percentages:
label
0    47.9
1    52.1
Name: proportion, dtype: float64

Testing label percentages:
label
0    47.9
1    52.1
Name: proportion, dtype: float64
